# H2 · Klasik makine ogrenmesi sablonu (herkes kendi algoritmasi)
1. Bu dosyayi Colab'da ac, **File > Save a copy in Drive** ile kendine kopyala.
2. Asagidaki hucrede `AD` ve `ALGORITMA` degerlerini kendine gore yaz.
3. **Runtime > Run all.** Sonuclar Drive'daki ortak `results/sonuclar.csv` dosyasina otomatik eklenir.

| Kisi | ALGORITMA |
|---|---|
| Fatih Kaya | `"rf"` Random Forest |
| Taha Yasir Guzeldal | `"svm"` Destek Vektor Makinesi (RBF) |
| Yunus Emir Gemici | `"knn"` K-En Yakin Komsu |
| Mumun Can Yildiz | `"lr"` Lojistik Regresyon |
| Mehmet Selim Ozdemir | `"gb"` Gradient Boosting (HistGradientBoosting) |
| Muhammet Emin Zora | `"dt"` Karar Agaci |

Ek temel model (istege bagli): `"nb"` Gaussian Naive Bayes.

In [ ]:
AD = "Fatih Kaya"        # <- kendi adin
ALGORITMA = "rf"         # <- rf | svm | knn | lr | gb | dt | nb
TOHUMLAR = [42]          # nihai sonuc icin [42, 0, 123]

In [ ]:
# KURULUM: Colab'da Drive'i baglar, depoyu GitHub'dan ceker. (Yerelde test icin YSA_PROJE_YOLU kullanilir.)
import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run("rm -rf /content/ysa-wifi-csi && git clone -q https://github.com/fat1hkaya/ysa-wifi-csi /content/ysa-wifi-csi",
                   shell=True, check=True)
    sys.path.insert(0, "/content/ysa-wifi-csi/src")
except ImportError:
    sys.path.insert(0, os.environ.get("YSA_SRC", "../src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ortak, ozellik
from ortak import SINIFLAR, DUSME
print("Veri yolu:", ortak.VERI_YOLU)

In [ ]:
# Oznitelikleri Drive'daki onbellekten oku; yoksa hesapla ve kaydet (~1 dk)
ONBELLEK = f"{ortak.PROJE_YOLU}/data/ozellik_uthar.npz"
if os.path.exists(ONBELLEK):
    d = np.load(ONBELLEK, allow_pickle=True)
    F = {k: d[k] for k in ["F_train", "F_val", "F_test", "y_train", "y_val", "y_test"]}
    isimler = list(d["isimler"])
    print("Onbellekten okundu:", ONBELLEK)
else:
    v = ortak.uthar_yukle()
    F = {}
    for b in ["train", "val", "test"]:
        F[f"F_{b}"], isimler = ozellik.ozellik_cikar(v[f"X_{b}"])
        F[f"y_{b}"] = v[f"y_{b}"]
    np.savez_compressed(ONBELLEK, isimler=np.array(isimler), **F)
    del v
    print("Hesaplandi ve kaydedildi:", ONBELLEK)
print({k: x.shape for k, x in F.items()})

## Model ve denenecek hiperparametreler
Her model `StandardScaler + model` boru hattidir; olcekleme **yalnizca egitim verisinden** ogrenilir.
Hiperparametreler **dogrulama (val)** kumesinde macro-F1'e gore secilir; **test kumesine yalnizca en sonda bir kez** bakilir.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

def model_kur(alg, p, tohum):
    if alg == "lr":  m = LogisticRegression(C=p["C"], max_iter=3000, class_weight="balanced")
    elif alg == "knn": m = KNeighborsClassifier(n_neighbors=p["k"], weights=p["agirlik"])
    elif alg == "svm": m = SVC(C=p["C"], gamma=p["gamma"], kernel="rbf", class_weight="balanced",
                               probability=p.get("olasilik", False), random_state=tohum)
    elif alg == "dt":  m = DecisionTreeClassifier(max_depth=p["derinlik"], min_samples_leaf=p["yaprak"],
                                                  class_weight="balanced", random_state=tohum)
    elif alg == "rf":  m = RandomForestClassifier(n_estimators=300, max_depth=p["derinlik"], max_features=p["oz"],
                                                  class_weight="balanced", n_jobs=-1, random_state=tohum)
    elif alg == "gb":  m = HistGradientBoostingClassifier(learning_rate=p["lr"], max_leaf_nodes=p["yaprak"], max_iter=200, max_features=0.3,
                                                          class_weight="balanced", early_stopping=True, random_state=tohum)
    elif alg == "nb":  m = GaussianNB(var_smoothing=p["vs"])
    else: raise ValueError(ALGORITMA)
    return make_pipeline(StandardScaler(), m)

IZGARA = {
    "lr":  {"C": [0.01, 0.1, 1, 10]},
    "knn": {"k": [1, 3, 5, 9, 15], "agirlik": ["uniform", "distance"]},
    "svm": {"C": [1, 10, 100], "gamma": ["scale", 1e-3, 1e-4]},
    "dt":  {"derinlik": [5, 10, 20, None], "yaprak": [1, 5, 10]},
    "rf":  {"derinlik": [None, 20], "oz": ["sqrt", 0.1]},
    "gb":  {"lr": [0.1], "yaprak": [15, 31]},          # gb yavastir: Colab'da toplam ~5-10 dk
    "nb":  {"vs": [1e-9, 1e-6, 1e-3]},
}
print(ALGORITMA, "->", len(list(ParameterGrid(IZGARA[ALGORITMA]))), "kombinasyon")

## Gorev 1 · 7 sinif aktivite tanima

In [ ]:
import time, json
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
Xtr, Xva, Xte = F["F_train"], F["F_val"], F["F_test"]
ytr, yva, yte = F["y_train"], F["y_val"], F["y_test"]

tohum = TOHUMLAR[0]
denemeler = []
for p in ParameterGrid(IZGARA[ALGORITMA]):
    t = time.time(); m = model_kur(ALGORITMA, p, tohum).fit(Xtr, ytr)
    f1 = f1_score(yva, m.predict(Xva), average="macro")
    denemeler.append((f1, p)); print(f"val macro-F1={f1:.4f}  {p}  ({time.time() - t:.1f} sn)")
en_iyi_f1, EN_IYI = max(denemeler, key=lambda d: d[0])
print("\nSecilen:", EN_IYI, f"(val macro-F1={en_iyi_f1:.4f})")

In [ ]:
for tohum in TOHUMLAR:
    m = model_kur(ALGORITMA, EN_IYI, tohum).fit(Xtr, ytr)
    t = time.time(); tahmin = m.predict(Xte); ms = 1000 * (time.time() - t) / len(Xte)
    s = ortak.degerlendir(yte, tahmin)
    print(f"tohum={tohum}  TEST accuracy={s['accuracy']:.4f}  macro-F1={s['macro_f1']:.4f}  ({ms:.3f} ms/ornek)")
    ortak.sonuc_kaydet({"deney": f"H2_ML_{ALGORITMA}", "gorev": "7sinif", "model": ALGORITMA, "tohum": tohum,
                        "accuracy": round(s["accuracy"], 4), "macro_f1": round(s["macro_f1"], 4),
                        "parametre": json.dumps(EN_IYI), "cikarim_ms": round(ms, 4), "yazan": AD})
print(classification_report(yte, tahmin, target_names=SINIFLAR, digits=3))
ConfusionMatrixDisplay(s["karisiklik"], display_labels=SINIFLAR).plot(xticks_rotation=45, cmap="Blues")
plt.title(f"{ALGORITMA} - test karisiklik matrisi"); plt.show()
model_7 = m

## Gorev 2 · Dusme / dusme degil (ikili)
Ayni hiperparametrelerle ikili model egitilir. Esik 0.5 yerine **dogrulama kumesinde** secilir:
dusme recall'u en az %90 olan en yuksek esik (kacirilan dusme, yanlis alarmdan daha pahali).

In [ ]:
ytr2, yva2, yte2 = ortak.ikili_etiket(ytr), ortak.ikili_etiket(yva), ortak.ikili_etiket(yte)
p2 = dict(EN_IYI, olasilik=True) if ALGORITMA == "svm" else EN_IYI
m2 = model_kur(ALGORITMA, p2, TOHUMLAR[0]).fit(Xtr, ytr2)
skor = lambda X: m2.predict_proba(X)[:, 1]

from sklearn.metrics import recall_score, precision_score
esikler = np.round(np.arange(0.05, 0.96, 0.05), 2)
sv = skor(Xva)
satirlar = [(e, recall_score(yva2, sv >= e, zero_division=0), precision_score(yva2, sv >= e, zero_division=0)) for e in esikler]
display(pd.DataFrame(satirlar, columns=["esik", "val_recall", "val_precision"]).round(3).set_index("esik").T)
uygun = [e for e, r, _ in satirlar if r >= 0.90]
ESIK = max(uygun) if uygun else 0.5
print("Secilen esik:", ESIK)

for esik in sorted({0.5, ESIK}):
    t = time.time(); tahmin2 = (skor(Xte) >= esik).astype(int); ms = 1000 * (time.time() - t) / len(Xte)
    s2 = ortak.degerlendir(yte2, tahmin2, ikili=True)
    print(f"esik={esik}: TEST dusme recall={s2['dusme_recall']:.3f} precision={s2['dusme_precision']:.3f} "
          f"F1={s2['dusme_f1']:.3f}  macro-F1={s2['macro_f1']:.3f}")
    ortak.sonuc_kaydet({"deney": f"H2_ML_{ALGORITMA}", "gorev": "ikili_dusme", "model": ALGORITMA, "tohum": TOHUMLAR[0],
                        "esik": esik, "accuracy": round(s2["accuracy"], 4), "macro_f1": round(s2["macro_f1"], 4),
                        "dusme_recall": round(s2["dusme_recall"], 4), "dusme_precision": round(s2["dusme_precision"], 4),
                        "dusme_f1": round(s2["dusme_f1"], 4), "parametre": json.dumps(EN_IYI),
                        "cikarim_ms": round(ms, 4), "yazan": AD})

## Model ne ogrendi? (oznitelik onemi - destekleyen modellerde)

In [ ]:
# 1) Grup bazli permutasyon onemi (tum modellerde ayni yontem, ~1 dk):
#    bir oznitelik grubunu karistirinca val macro-F1 ne kadar dusuyor?
gruplar = {}
for i, n in enumerate(isimler):
    g = n.split("__")[1] if "__" in n else ("fft_bant" if n.startswith("fft") else "hareket_dilim")
    gruplar.setdefault(g, []).append(i)
for a_ in range(3):
    gruplar[f"anten{a_ + 1}"] = [i for i, n in enumerate(isimler) if n.startswith(f"a{a_ + 1}_")]
temel = f1_score(yva, model_7.predict(Xva), average="macro")
rng = np.random.default_rng(0); dusus = {}
for g, idx in gruplar.items():
    Xk = Xva.copy(); Xk[:, idx] = Xk[rng.permutation(len(Xk))][:, idx]
    dusus[g] = temel - f1_score(yva, model_7.predict(Xk), average="macro")
pd.Series(dusus).sort_values().plot.barh(figsize=(6, 4.5), title=f"{ALGORITMA}: grup karistirilinca macro-F1 dususu")
plt.xlabel("macro-F1 dususu (buyuk = onemli)"); plt.show()

# 2) Modelin kendi onem degerleri (yalnizca rf, dt, lr)
son = model_7[-1]
onem = son.feature_importances_ if hasattr(son, "feature_importances_") else (
       np.abs(son.coef_).mean(0) if hasattr(son, "coef_") else None)
if onem is not None:
    sira = np.argsort(onem)[::-1][:15]
    display(pd.DataFrame({"oznitelik": [isimler[i] for i in sira], "onem": np.round(onem[sira], 5)}))

In [ ]:
# Ortak sonuc tablosu (herkesin satirlari)
display(pd.read_csv(ortak.SONUC_YOLU).tail(20))

## Gruba / rapora yazilacaklar
- Algoritma ve secilen hiperparametreler, val macro-F1
- Test: accuracy, macro-F1, en cok karisan iki sinif (karisiklik matrisinden)
- Dusme: esik 0.5 ve secilen esikte recall/precision
- En onemli 3 oznitelik turu ve yorumu